In [3]:
import geopandas as gpd
import pandas as pd 
import numpy as np
from pathlib import Path
from shapely import wkt

In [57]:
rc_lookups = pd.read_csv("output/rc_lookups.csv")
rc_ldc =pd.read_csv("output/retail_centers_with_ldc_retailN.csv")

In [58]:
rc = rc_ldc.merge(
     rc_lookups.drop(columns=['geometry']),
    on='RC_ID',
    how='left'
)
rc

,RC_ID,Total_POI_count,H3_cells,Retail,Dining_and_Drinking,Leisure_Travel,Community_Health,category_names,centre_type,LocalAuthorityNM_1,LocalAuthorityNM_2,LocalAuthorityNM_3,geometry,LDC_retailN,LAD24NM,CountyName,RegionName,gsscode
0,6025,16,5,7.0,6.0,0.0,3.0,['Asian Restaurant' 'Electronics Store' 'Café'...,centre,North East Derbyshire,0,0,MULTIPOLYGON (((440445.7388613202 367018.09139...,2.0,North East Derbyshire,Derbyshire,East Midlands,E63008766
1,1193,131,71,86.0,33.0,4.0,8.0,"['Electronics Store' ""Doctor's Office"" 'Drugst...",centre,North East Derbyshire,0,0,MULTIPOLYGON (((439116.7765939621 363407.88751...,131.0,North East Derbyshire,Derbyshire,East Midlands,E63008809
2,6644,14,11,11.0,2.0,1.0,0.0,['Electronics Store' 'Hardware Store' 'Conveni...,centre,Cornwall,0,0,MULTIPOLYGON (((194309.88097327633 76200.71289...,10.0,Cornwall,Cornwall,South West,E63013978
3,6026,16,44,11.0,3.0,2.0,0.0,['Gym' 'Gastropub' 'Brewery' 'Funeral Home' 'B...,centre,North East Derbyshire,0,0,MULTIPOLYGON (((439008.4176877096 364010.85388...,17.0,North East Derbyshire,Derbyshire,East Midlands,E63008809
4,7407,12,4,6.0,3.0,1.0,2.0,"['Hardware Store' ""Doctor's Office"" 'Indian Re...",centre,Chesterfield,0,0,MULTIPOLYGON (((438294.8993448653 371476.61951...,8.0,Chesterfield,Derbyshire,East Midlands,E63008717
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9617,2844,46,16,32.0,13.0,1.0,0.0,['Convenience Store' 'Discount Store' 'Restaur...,retail_park,"Armagh City, Banbridge and Craigavon",0,0,MULTIPOLYGON (((111614.72348488995 512113.4056...,59.0,"Armagh City, Banbridge and Craigavon",Armagh,NI,NI
9618,4025,29,22,17.0,8.0,2.0,2.0,['Drugstore' 'Pet Service' 'Convenience Store'...,retail_park,Braintree,0,0,MULTIPOLYGON (((578163.6762067255 233984.52018...,14.0,Braintree,Essex,East of England,E63011054
9619,2676,50,11,34.0,14.0,2.0,0.0,['Textiles Store' 'Electronics Store' 'Sportin...,retail_park,"Bournemouth, Christchurch and Poole",0,0,MULTIPOLYGON (((401222.3074132583 90489.386812...,46.0,"Bournemouth, Christchurch and Poole",Dorset,South West,E63013860
9620,3757,32,22,26.0,6.0,0.0,0.0,['Sporting Goods Retail' 'Convenience Store' '...,retail_park,Glasgow City,0,0,MULTIPOLYGON (((255356.56308415128 666668.6817...,50.0,Glasgow City,City of glasgow,Scotland,S45001180


# apply_hierarchical_classification

In [169]:
def apply_hierarchical_classification(df: pd.DataFrame, retail_park_rc_ids=None) -> pd.DataFrame:
    df = df.copy()
    retail_park_rc_ids = set() if retail_park_rc_ids is None else set(retail_park_rc_ids)

    # 1. HANDLE NA VALUES PROPERLY
    # Instead of filling with "NA" before ranking, we rank first 
    # and then handle the NAs to match the R script logic.
    rank_specs = {
        "RGN_rank": "RegionName",
        "CTY_rank": "CountyName",
        "LAD_rank": "LAD24NM",
        "BUA_rank": "gsscode",
    }
    
    for rank_col, group_col in rank_specs.items():
        # Only rank where the group name is NOT null
        mask = df[group_col].notna()
        df.loc[mask, rank_col] = df[mask].groupby(group_col)["LDC_retailN"].rank(
            method="first", ascending=False
        )
        # If it was NA, give it a massive rank so it's never Rank 1
        df[rank_col] = df[rank_col].fillna(999999)

    df["tr_retailN"] = df["LDC_retailN"]
    df["Classification"] = pd.Series(pd.NA, index=df.index)

    def assign(label: str, mask: pd.Series) -> None:
        df.loc[df["Classification"].isna() & mask, "Classification"] = label

    # TIER 1: Regional (With Manual Overrides)
    assign("Regional Centre", (df["RGN_rank"] == 1))

    # TIER 2: Major Town
    assign("Major Town Centre", (df["CTY_rank"] == 1) & (df["RGN_rank"] != 1) & (df["LDC_retailN"] >= 350))

    # TIER 3: Town Centre
    assign("Town Centre", 
        ((df["CTY_rank"] == 1) & (df["RGN_rank"] != 1)) | 
        ((df["LAD_rank"] == 1) & (df["RGN_rank"] != 1) & (df["CTY_rank"] != 1)) | 
        ((df["LDC_retailN"] >= 250) & (df["LDC_retailN"] < 350)) 
    )

    # TIER 4: Market Town
    assign("Market Town", 
        (df["BUA_rank"] == 1) & (df["LAD_rank"] != 1) & (df["RGN_rank"] != 1) & (df["CTY_rank"] != 1) & 
        (df["tr_retailN"] >= 150) & (df["tr_retailN"] < 250)
    )

    # TIER 5: District Centre
    assign("District Centre", 
        (df["BUA_rank"] != 1) & (df["LAD_rank"] != 1) & (df["RGN_rank"] != 1) & (df["CTY_rank"] != 1) & 
        (df["tr_retailN"] >= 150)
    )

    # TIER 6: Local Centre
    assign("Local Centre", df["tr_retailN"] < 150)

    # REFINEMENT: Local -> Small Local
    df.loc[(df["Classification"] == "Local Centre") & (df["tr_retailN"] < 100), "Classification"] = "Small Local Centre"

    # FALLBACKS
    df.loc[df["Classification"].isna() & df["LDC_retailN"].notna(), "Classification"] = "Market Town"
    df["Classification"] = df["Classification"].fillna("Small Local Centre")

    # RETAIL PARK ADJUSTMENT
    # Refined to include District Centres if they are actually Retail Parks
    retail_park_mask = (
        (df["centre_type"] == "retail_park") 
    ) & (df["Classification"].isin(["Small Local Centre"]))
    # retail_park_mask = (
    #     (df["centre_type"] == "retail_park") | 
    #     df["RC_ID"].isin(retail_park_rc_ids)
    # ) & (df["Classification"].isin(["Small Local Centre", "Local Centre", "District Centre"]))
    
    df.loc[retail_park_mask, "Classification"] = "Retail Park"
    df.loc[(df["Classification"] == "Retail Park") & (df["tr_retailN"] < 50), "Classification"] = "Small Retail Park"
    df.loc[(df["Classification"] == "Retail Park") & (df["tr_retailN"] >= 50), "Classification"] = "Large Retail Park"

    return df.sort_values(["RegionName", "RC_ID"]).reset_index(drop=True)

In [176]:
df= apply_hierarchical_classification(rc)

In [180]:
df[df['RC_ID'] == 8]

,RC_ID,Total_POI_count,H3_cells,Retail,Dining_and_Drinking,Leisure_Travel,Community_Health,category_names,centre_type,LocalAuthorityNM_1,...,LAD24NM,CountyName,RegionName,gsscode,RGN_rank,CTY_rank,LAD_rank,BUA_rank,tr_retailN,Classification
7499,8,1496,213,754.0,581.0,138.0,23.0,['Electronics Store' 'Tourist Information and ...,centre,Cardiff,...,Cardiff,South glamorgan,Wales,W37000384,1.0,1.0,1.0,1.0,1001.0,Regional Centre


# add manual

In [143]:
def apply_manual_edits(
    df: pd.DataFrame,
    rcid_lookup_csv: str = "output/rcid_lookup_old_to_new.csv",
    manual_hierarchy_csv: str | None = None,
) -> pd.DataFrame:
    """
    Manual classification edits + optional Manual_Hierarchy upgrades.
    rcid_lookup_csv: path to rcid_lookup_old_to_new.csv
    manual_hierarchy_csv: path to Manual_Hierarchy.csv (optional)
    """
    df = df.copy()

    # --- RC_ID mapping (old -> new) ---
    lookup = pd.read_csv(rcid_lookup_csv)
    old_to_new = dict(zip(lookup["RC_ID"], lookup["new_rcid"]))

    def map_rcid(old_id: str) -> str:
        return old_to_new.get(old_id, old_id)

    # --- Manual classification edits (old RC_IDs in the R code) ---
    manual_classification_old = {
        "RC_EW_2920": "Market Town",
        "RC_SC_332": "Major Town Centre",
        "RC_EW_876": "Major Town Centre",
        "RC_EW_5564": "Major Town Centre",
        "RC_EW_61": "Major Town Centre",
        "RC_EW_3096": "Major Town Centre",
        "RC_EW_3254": "Major Town Centre",
        "RC_EW_3301": "Major Town Centre",
        "RC_EW_3476": "Major Town Centre",
        "RC_EW_3522": "Regional Centre", # plus bristol
        "RC_EW_2751": "Regional Centre", # plus liverpool one
         "RC_SC_280": "Regional Centre", # plus edinburgh
    }

    manual_classification = {map_rcid(k): v for k, v in manual_classification_old.items()}
    for rcid, cls in manual_classification.items():
        df.loc[df["RC_ID"] == rcid, "Classification"] = cls

    # --- Manual hierarchy list (upgrade Town Centre -> Major Town Centre) ---
    if manual_hierarchy_csv:
        manual_path = Path(manual_hierarchy_csv)
        if manual_path.exists():
            rc_manual = pd.read_csv(manual_path)
            rc_manual = rc_manual.loc[
                rc_manual["H_Classification"] == "Major Town Centre", "RC_ID"
            ].dropna()
            rc_manual = [map_rcid(x) for x in rc_manual.astype(str).unique()]

            mask = (df["RC_ID"].isin(rc_manual)) & (df["Classification"] == "Town Centre")
            df.loc[mask, "Classification"] = "Major Town Centre"

    return df

In [181]:
df1 = apply_manual_edits(df)
df1[df1['RC_ID'] == 8]

,RC_ID,Total_POI_count,H3_cells,Retail,Dining_and_Drinking,Leisure_Travel,Community_Health,category_names,centre_type,LocalAuthorityNM_1,...,LAD24NM,CountyName,RegionName,gsscode,RGN_rank,CTY_rank,LAD_rank,BUA_rank,tr_retailN,Classification
7499,8,1496,213,754.0,581.0,138.0,23.0,['Electronics Store' 'Tourist Information and ...,centre,Cardiff,...,Cardiff,South glamorgan,Wales,W37000384,1.0,1.0,1.0,1.0,1001.0,Regional Centre


# shopping centres

In [144]:
def apply_shopping_outlet_overrides(
    df: pd.DataFrame,
    rc_uk_path: str = "geo/postworkshopcentres.gpkg",
    rcid_lookup_csv: str = "output/rcid_lookup_old_to_new.csv",
) -> pd.DataFrame:
    """
    Manual outlet locations (spatial intersection + explicit ID overrides).
    rc_uk_path: path to postworkshopcentres.gpkg
    rcid_lookup_csv: path to rcid_lookup_old_to_new.csv
    """
    df = df.copy()

    # --- RC_ID mapping (old -> new) ---
    lookup = pd.read_csv(rcid_lookup_csv)
    old_to_new = dict(zip(lookup["RC_ID"], lookup["new_rcid"]))

    def map_rcid(old_id: str) -> str:
        return old_to_new.get(old_id, old_id)

    # --- Manual outlet locations ---
    outlets = pd.DataFrame(
        [
            ("Regional Shopping Centre", "Westfield London", 51.50768597440922, -0.22088352838801395),
            ("Regional Shopping Centre", "Metrocentre", 54.957214904105754, -1.6674536751658506),
            ("Regional Shopping Centre", "Trafford Centre", 53.465100417291744, -2.347637851106119),
            ("Regional Shopping Centre", "Westfield Stratford City", 51.54425552636472, -0.004332458383472109),
            ("Regional Shopping Centre", "Bluewater", 51.43912203176112, 0.27160383738290644),
            ("Regional Shopping Centre", "intu Merry Hill", 52.48138718476229, -2.110658395511001),
            ("Regional Shopping Centre", "Lakeside Shopping Centre", 51.489406902239956, 0.2853764470235825),
            ("Regional Shopping Centre", "Meadowhall", 53.41411230589209, -1.4111151687565566),
            ("Regional Shopping Centre", "East Kilbride Shopping Centre", 55.76021412051323, -4.17708950101207),
            ("Regional Shopping Centre", "Silverburn Centre", 55.82112954772368, -4.341253330960348),
            ("Regional Shopping Centre", "intu Braehead", 55.87600450967236, -4.364642953985305),
            ("Regional Shopping Centre", "White Rose Centre", 53.75868081616114, -1.5734907774777223),
            ("Large Outlet Shopping Centre", "Ashford Designer Outlet", 51.137387301260496, 0.8801687441992894),
            ("Large Outlet Shopping Centre", "Astle Park West Bromwich", 52.520103339621166, -1.9960259925402608),
            ("Large Outlet Shopping Centre", "Bicester Outlet Village", 51.89237714202758, -1.1560770883204268),
            ("Large Outlet Shopping Centre", "Bideford Atlantic Village", 51.00755545946937, -4.237327761297502),
            #("Large Outlet Shopping Centre", "Burberry Factory Outlet London", 51.54651001026377, -0.05093197052890261),
            ("Large Outlet Shopping Centre", "Castleford Junction 32 Outlet", 53.712463797168354, -1.3388364244282458),
            ("Large Outlet Shopping Centre", "Chatham Dockside Outlet Centre", 51.40160304040388, 0.5377104489852201),
            ("Large Outlet Shopping Centre", "Cheshire Oaks Designer Outlet", 53.26539272808919, -2.8809029974792315),
            ("Large Outlet Shopping Centre", "Clacton Factory Outlet", 51.81858769266023, 1.168209463308007),
            ("Large Outlet Shopping Centre", "Clarks Village Outlet Somerset", 51.12833201493458, -2.741588552823589),
            ("Large Outlet Shopping Centre", "Dalton Park Outlet", 54.81537131857434, -1.3739226162513756),
            ("Large Outlet Shopping Centre", "Doncaster Lakeside Village", 53.50769203205623, -1.1135984485990036),
            ("Large Outlet Shopping Centre", "Dover De Bradelei Wharf Outlet", 51.12211019226935, 1.3141697547138849),
            ("Large Outlet Shopping Centre", "East Midlands Outlet Junction 28", 53.108083884731556, -1.3126583496100666),
            ("Large Outlet Shopping Centre", "Freeport Braintree", 51.870070700810075, 0.5714801267433955),
            ("Large Outlet Shopping Centre", "Freeport Fleetwood Outlet", 53.91786152727727, -3.0118505404061744),
            ("Large Outlet Shopping Centre", "Freeport Talke Stoke-on-Trent", 53.070597467076176, -2.263078253210015),
            ("Large Outlet Shopping Centre", "Gloucester Quays Outlet", 51.85953205593224, -2.2516156614946916),
            ("Large Outlet Shopping Centre", "Hertfordshire Galleria Outlet Shopping", 51.76141229905345, -0.24021860677316503),
            ("Large Outlet Shopping Centre", "Hornsea Freeport Outlet", 53.898485374967684, -0.1724229262407349),
            ("Large Outlet Shopping Centre", "Kilver Court Designer Village", 51.19063584650441, -2.5353243560244905),
            ("Large Outlet Shopping Centre", "London Designer Outlet Wembley", 51.55646778421662, -0.28320440479433673),
            ("Large Outlet Shopping Centre", "Lowry Outlet Salford Quays", 53.47092253822526, -2.292514874626797),
            ("Large Outlet Shopping Centre", "Newcastle Royal Quays Outlet", 54.99532901611218, -1.4642812945442476),
            ("Large Outlet Shopping Centre", "Portsmouth Gunwharf Quays", 50.79546395254288, -1.103824062300293),
            ("Large Outlet Shopping Centre", "Springfields Spalding Outlet", 52.799438891276665, -0.126329033767297),
            ("Large Outlet Shopping Centre", "Swindon Designer Outlet", 51.56272470333755, -1.7971899666540831),
            ("Large Outlet Shopping Centre", "The Mill Outlet Batley", 53.7129199988173, -1.6251337290018772),
            ("Large Outlet Shopping Centre", "York Designer Outlet", 53.92233415376806, -1.0768705880781113),
            ("Large Outlet Shopping Centre", "Banbridge Village Outlet", 54.33172185692118, -6.276147653115943),
            ("Large Outlet Shopping Centre", "Cribbs Causeway", 51.524552488774034, -2.5955220725284307),
            ("Large Outlet Shopping Centre", "Bridgend Designer Outlets Wales", 51.532410963682594, -3.576839020076668),
            ("Large Outlet Shopping Centre", "Gretna Gateway Outlet Village", 54.99688074646182, -3.056523264134829),
            ("Large Outlet Shopping Centre", "Livingston Designer Outlet", 55.88310933797727, -3.520137944032474),
            ("Large Outlet Shopping Centre", "Sterling Mills Outlet", 56.14927287877639, -3.740053433726151),
        ],
        columns=["SubType", "Name", "Lat", "Lon"],
    )
    outlets["Type"] = "Out of Town Shopping Centres"

    outlets_gdf = gpd.GeoDataFrame(
        outlets,
        geometry=gpd.points_from_xy(outlets["Lon"], outlets["Lat"]),
        crs="EPSG:4326",
    )

    rc_uk = gpd.read_file(rc_uk_path)
    outlets_gdf = outlets_gdf.to_crs(rc_uk.crs)

    outlets_joined = gpd.sjoin(
        outlets_gdf,
        rc_uk[["RC_ID", "geometry"]],
        how="left",
        predicate="intersects",
    )
    outlets_tag = outlets_joined[["Type", "RC_ID"]].dropna(subset=["RC_ID"])

    manual_outlet_rcids_old = {
        "Ashford Designer Outlet": "RC_EW_1312",
        "Chatham Dockside Outlet Centre": "RC_EW_1529",
        "Doncaster Lakeside Village": "RC_EW_762",
        "Swindon Designer Outlet": "RC_EW_3751",
        "Springfields Spalding Outlet": "RC_EW_955",
    }

    manual_override = outlets_gdf[outlets_gdf["Name"].isin(manual_outlet_rcids_old)].copy()
    manual_override["RC_ID"] = manual_override["Name"].map(manual_outlet_rcids_old).map(map_rcid)
    manual_override = manual_override[["Type", "RC_ID"]].dropna(subset=["RC_ID"])

    outlets_tag = pd.concat([outlets_tag, manual_override], ignore_index=True)

    df = df.merge(outlets_tag, on="RC_ID", how="left", sort=False)
    df.loc[df["Type"].notna(), "Classification"] = df.loc[df["Type"].notna(), "Type"]
    df = df.drop(columns=["Type"])

    return df


In [182]:
df2 = apply_shopping_outlet_overrides(df1)
df2[df2['RC_ID'] == 8]

,RC_ID,Total_POI_count,H3_cells,Retail,Dining_and_Drinking,Leisure_Travel,Community_Health,category_names,centre_type,LocalAuthorityNM_1,...,LAD24NM,CountyName,RegionName,gsscode,RGN_rank,CTY_rank,LAD_rank,BUA_rank,tr_retailN,Classification
7501,8,1496,213,754.0,581.0,138.0,23.0,['Electronics Store' 'Tourist Information and ...,centre,Cardiff,...,Cardiff,South glamorgan,Wales,W37000384,1.0,1.0,1.0,1.0,1001.0,Regional Centre


In [145]:
def apply_out_of_town_sizing(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    out_of_town_mask = df["Classification"].isin(["Out of Town Shopping Centres"])

    df.loc[out_of_town_mask & (df["tr_retailN"] < 50), "Classification"] = "Small Shopping Centre"
    df.loc[out_of_town_mask & (df["tr_retailN"] >= 50), "Classification"] = "Large Shopping Centre"

    return df


In [183]:
df3 = apply_out_of_town_sizing(df2)
df3[df3['RC_ID'] == 8]

,RC_ID,Total_POI_count,H3_cells,Retail,Dining_and_Drinking,Leisure_Travel,Community_Health,category_names,centre_type,LocalAuthorityNM_1,...,LAD24NM,CountyName,RegionName,gsscode,RGN_rank,CTY_rank,LAD_rank,BUA_rank,tr_retailN,Classification
7501,8,1496,213,754.0,581.0,138.0,23.0,['Electronics Store' 'Tourist Information and ...,centre,Cardiff,...,Cardiff,South glamorgan,Wales,W37000384,1.0,1.0,1.0,1.0,1001.0,Regional Centre


In [ ]:
df3['Classification'].isna().sum()

In [ ]:
cols = ["RGN_rank", "CTY_rank", "LAD_rank", "BUA_rank", "tr_retailN"]
df3[cols] = df3[cols].apply(pd.to_numeric, errors="coerce").round().astype("int64")

In [ ]:
df3["geometry"] = df3["geometry"].apply(
    lambda geom: wkt.loads(geom) if isinstance(geom, str) else geom
)

# Now create GeoDataFrame with CRS
gdf = gpd.GeoDataFrame(df3, geometry="geometry", crs="EPSG:27700")  # change CRS if needed
hex_map = {
    "Regional Centre": "#882255",
    "Major Town Centre": "#332288",
    "Town Centre": "#117733",
    "Market Town": "#44AA99",
    "District Centre": "#999933",
    "Local Centre": "#DDCC77",
    "Small Local Centre": "#88CCEE",
    "Large Retail Park": "#CC6677",
    "Small Retail Park": "#E0A1AB",
    "Large Out of Town Shopping Centre": "#AA4499",
    "Small Out of Town Shopping Centre": "#CD83C1",
}

gdf["HEX_COL"] = gdf["Classification"].map(hex_map)

# Export (pick your path)
out_path = "output/rc2025_classified.shp"
gdf.to_file(out_path, driver="ESRI Shapefile")

In [ ]:
geo = df3
geo["geometry"] = geo["geometry"].apply(
    lambda geom: wkt.loads(geom) if isinstance(geom, str) else geom
)

geo_gdf = gpd.GeoDataFrame(geo, geometry="geometry", crs="EPSG:27700")
geo_gdf.to_file("output/rc2025_classified.gpkg", driver="GPKG")


# Join the split togther and disolve the build up area

In [135]:
# rc_geo = gpd.read_file("output/rc2025_classified.gpkg")
rc_ldc =pd.read_csv("output/retail_centers_with_ldc_retailN.csv")
rc_lookups = pd.read_csv("output/rc_lookups_2011BUA.csv") # 2011 BUA version
rc = rc_ldc.merge(
     rc_lookups.drop(columns=['geometry']),
    on='RC_ID',
    how='left'
)

rc["geometry"] = rc["geometry"].apply(
    lambda geom: wkt.loads(geom) if isinstance(geom, str) else geom
)

rc_geo = gpd.GeoDataFrame(
    rc,
    geometry="geometry",
    crs="EPSG:4326"  # set to the correct CRS for your coordinates
)
rc_geo.rename(columns={"LDC_retailN": "tr_retailN"}, inplace=True)
rc_geo

,RC_ID,Total_POI_count,H3_cells,Retail,Dining_and_Drinking,Leisure_Travel,Community_Health,category_names,centre_type,LocalAuthorityNM_1,LocalAuthorityNM_2,LocalAuthorityNM_3,geometry,tr_retailN,LAD24NM,CountyName,RegionName,gsscode
0,6025,16,5,7.0,6.0,0.0,3.0,['Asian Restaurant' 'Electronics Store' 'Café'...,centre,North East Derbyshire,0,0,"MULTIPOLYGON (((440445.73886 367018.09139, 440...",2.0,North East Derbyshire,Derbyshire,East Midlands,E34004845
1,1193,131,71,86.0,33.0,4.0,8.0,"['Electronics Store' ""Doctor's Office"" 'Drugst...",centre,North East Derbyshire,0,0,"MULTIPOLYGON (((439116.77659 363407.88751, 439...",131.0,North East Derbyshire,Derbyshire,East Midlands,E34004845
2,6644,14,11,11.0,2.0,1.0,0.0,['Electronics Store' 'Hardware Store' 'Conveni...,centre,Cornwall,0,0,"MULTIPOLYGON (((194309.88097 76200.71289, 1943...",10.0,Cornwall,Cornwall,South West,E34001781
3,6026,16,44,11.0,3.0,2.0,0.0,['Gym' 'Gastropub' 'Brewery' 'Funeral Home' 'B...,centre,North East Derbyshire,0,0,"MULTIPOLYGON (((439008.41769 364010.85389, 438...",17.0,North East Derbyshire,Derbyshire,East Midlands,E34004845
4,7407,12,4,6.0,3.0,1.0,2.0,"['Hardware Store' ""Doctor's Office"" 'Indian Re...",centre,Chesterfield,0,0,"MULTIPOLYGON (((438294.89934 371476.61952, 438...",8.0,Chesterfield,Derbyshire,East Midlands,E34004985
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9617,2844,46,16,32.0,13.0,1.0,0.0,['Convenience Store' 'Discount Store' 'Restaur...,retail_park,"Armagh City, Banbridge and Craigavon",0,0,"MULTIPOLYGON (((111614.72348 512113.40565, 111...",59.0,"Armagh City, Banbridge and Craigavon",Armagh,NI,NI
9618,4025,29,22,17.0,8.0,2.0,2.0,['Drugstore' 'Pet Service' 'Convenience Store'...,retail_park,Braintree,0,0,"MULTIPOLYGON (((578163.67621 233984.52019, 578...",14.0,Braintree,Essex,East of England,E34004351
9619,2676,50,11,34.0,14.0,2.0,0.0,['Textiles Store' 'Electronics Store' 'Sportin...,retail_park,"Bournemouth, Christchurch and Poole",0,0,"MULTIPOLYGON (((401222.30741 90489.38681, 4011...",46.0,"Bournemouth, Christchurch and Poole",Dorset,South West,E34005031
9620,3757,32,22,26.0,6.0,0.0,0.0,['Sporting Goods Retail' 'Convenience Store' '...,retail_park,Glasgow City,0,0,"MULTIPOLYGON (((255356.56308 666668.68178, 255...",50.0,Glasgow City,City of glasgow,Scotland,NaN


In [197]:
# split info
rc_split = gpd.read_file("geo/postworkshopcentres_withsplit.gpkg")
rc_split = gpd.read_file("geo/2postworkshopcentres.gpkg")
rc_split.head()
# # normalize new split format:
# rc_split.rename(columns={"RC_ID": "RC_ID_split"}, inplace=True)
# split_parts = rc_split["RC_ID_split"].astype(str).str.split("_", n=1, expand=True)
# split_parts
# rc_split["RC_ID"] = split_parts[1].fillna(split_parts[0]).astype(str)


,RC_ID,Total_POI_count,H3_cells,Retail,Dining_and_Drinking,Leisure_Travel,Community_Health,category_names,centre_type,LocalAuthorityNM_1,LocalAuthorityNM_2,LocalAuthorityNM_3,was_split,Original_POI_count,Parent_ID,geometry
0,6025,16,5,7.0,6.0,0.0,3.0,['Asian Restaurant' 'Electronics Store' 'Café'...,centre,North East Derbyshire,None,None,False,16,NaN,"MULTIPOLYGON (((440445.739 367018.091, 440420...."
1,1193,131,71,86.0,33.0,4.0,8.0,"['Electronics Store' ""Doctor's Office"" 'Drugst...",centre,North East Derbyshire,None,None,False,131,NaN,"MULTIPOLYGON (((439116.777 363407.888, 439111...."
2,6644,14,11,11.0,2.0,1.0,0.0,['Electronics Store' 'Hardware Store' 'Conveni...,centre,Cornwall,None,None,False,14,NaN,"MULTIPOLYGON (((194309.881 76200.713, 194331.3..."
3,6026,16,44,11.0,3.0,2.0,0.0,['Gym' 'Gastropub' 'Brewery' 'Funeral Home' 'B...,centre,North East Derbyshire,None,None,False,16,NaN,"MULTIPOLYGON (((439008.418 364010.854, 438983...."
4,7407,12,4,6.0,3.0,1.0,2.0,"['Hardware Store' ""Doctor's Office"" 'Indian Re...",centre,Chesterfield,None,None,False,12,NaN,"MULTIPOLYGON (((438294.899 371476.62, 438289.9..."


In [198]:
rc_split[rc_split['Parent_ID']==8]

,RC_ID,Total_POI_count,H3_cells,Retail,Dining_and_Drinking,Leisure_Travel,Community_Health,category_names,centre_type,LocalAuthorityNM_1,LocalAuthorityNM_2,LocalAuthorityNM_3,was_split,Original_POI_count,Parent_ID,geometry
2378,119,678,108,360.0,237.0,65.0,16.0,['Tourist Information and Service' 'Gastropub'...,centre,Islington,None,None,True,2691,8.0,"MULTIPOLYGON (((531523.843 183367.755, 531498...."
2379,200,502,106,241.0,196.0,55.0,10.0,"['Electronics Store' ""Doctor's Office"" 'Touris...",centre,Islington,None,None,True,2691,8.0,"MULTIPOLYGON (((530980.539 183158.83, 531005.8..."
2380,353,358,79,142.0,165.0,34.0,17.0,"['Latin American Restaurant' ""Doctor's Office""...",centre,Islington,None,None,True,2691,8.0,"MULTIPOLYGON (((531213.219 185081.154, 531208...."
2381,294,399,77,211.0,163.0,15.0,10.0,['Latin American Restaurant' 'Gastropub' 'Souv...,centre,Islington,None,None,True,2691,8.0,"MULTIPOLYGON (((530484.579 185894.784, 530479...."
2382,589,248,69,138.0,84.0,14.0,12.0,['Electronics Store' 'Gastropub' 'Souvlaki Sho...,centre,Islington,None,None,True,2691,8.0,"MULTIPOLYGON (((529176.313 186186.573, 529171...."
2383,1147,137,48,45.0,70.0,15.0,7.0,['Electronics Store' 'Sporting Goods Retail' '...,centre,Islington,None,None,True,2691,8.0,"MULTIPOLYGON (((530797.789 185321.114, 530818...."
2384,1056,149,38,92.0,40.0,9.0,8.0,"['Electronics Store' ""Doctor's Office"" 'Latin ...",centre,Islington,None,None,True,2691,8.0,"MULTIPOLYGON (((531874.711 183740.362, 531849...."
2385,1867,78,22,32.0,34.0,8.0,4.0,"[""Doctor's Office"" 'Betting Shop' 'Convenience...",centre,Islington,None,None,True,2691,8.0,"MULTIPOLYGON (((529938.015 186473.361, 529933...."
2386,1504,101,19,56.0,32.0,8.0,5.0,['Sushi Restaurant' 'Physical Therapy Clinic' ...,centre,Islington,None,None,True,2691,8.0,"MULTIPOLYGON (((531571.519 184258.771, 531591...."
2387,3050,41,18,11.0,15.0,12.0,3.0,"[""Doctor's Office"" 'Physical Therapy Clinic' '...",centre,Islington,None,None,True,2691,8.0,"MULTIPOLYGON (((531710.374 182851.754, 531705...."


In [195]:
rc_work[rc_work['group_id']==8]

,RC_ID,Total_POI_count,H3_cells,Retail,Dining_and_Drinking,Leisure_Travel,Community_Health,category_names,centre_type,LocalAuthorityNM_1,...,LocalAuthorityNM_3,geometry,tr_retailN,LAD24NM,CountyName,RegionName,gsscode,was_split,Parent_ID,group_id
2378,119,678,108,360.0,237.0,65.0,16.0,['Tourist Information and Service' 'Gastropub'...,centre,Islington,...,0,"MULTIPOLYGON (((531523.8433 183367.75465, 5314...",393.0,Islington,Greater london,London,E34004707,True,8.0,8
2379,200,502,106,241.0,196.0,55.0,10.0,"['Electronics Store' ""Doctor's Office"" 'Touris...",centre,Islington,...,0,"MULTIPOLYGON (((530980.53899 183158.83048, 531...",308.0,Islington,Greater london,London,E34004707,True,8.0,8
2380,353,358,79,142.0,165.0,34.0,17.0,"['Latin American Restaurant' ""Doctor's Office""...",centre,Islington,...,0,"MULTIPOLYGON (((531213.21867 185081.15414, 531...",220.0,Islington,Greater london,London,E34004707,True,8.0,8
2381,294,399,77,211.0,163.0,15.0,10.0,['Latin American Restaurant' 'Gastropub' 'Souv...,centre,Islington,...,0,"MULTIPOLYGON (((530484.57927 185894.78405, 530...",299.0,Islington,Greater london,London,E34004707,True,8.0,8
2382,589,248,69,138.0,84.0,14.0,12.0,['Electronics Store' 'Gastropub' 'Souvlaki Sho...,centre,Islington,...,0,"MULTIPOLYGON (((529176.31329 186186.57342, 529...",171.0,Islington,Greater london,London,E34004707,True,8.0,8
2383,1147,137,48,45.0,70.0,15.0,7.0,['Electronics Store' 'Sporting Goods Retail' '...,centre,Islington,...,0,"MULTIPOLYGON (((530797.78893 185321.11428, 530...",73.0,Islington,Greater london,London,E34004707,True,8.0,8
2384,1056,149,38,92.0,40.0,9.0,8.0,"['Electronics Store' ""Doctor's Office"" 'Latin ...",centre,Islington,...,0,"MULTIPOLYGON (((531874.71102 183740.36173, 531...",108.0,Islington,Greater london,London,E34004707,True,8.0,8
2385,1867,78,22,32.0,34.0,8.0,4.0,"[""Doctor's Office"" 'Betting Shop' 'Convenience...",centre,Islington,...,0,"MULTIPOLYGON (((529938.01546 186473.36052, 529...",55.0,Islington,Greater london,London,E34004707,True,8.0,8
2386,1504,101,19,56.0,32.0,8.0,5.0,['Sushi Restaurant' 'Physical Therapy Clinic' ...,centre,Islington,...,0,"MULTIPOLYGON (((531571.51885 184258.77123, 531...",57.0,Islington,Greater london,London,E34004707,True,8.0,8
2387,3050,41,18,11.0,15.0,12.0,3.0,"[""Doctor's Office"" 'Physical Therapy Clinic' '...",centre,Islington,...,0,"MULTIPOLYGON (((531710.37384 182851.75401, 531...",15.0,Islington,Greater london,London,E34004707,True,8.0,8


In [193]:
rc_work = rc_geo.copy()

# --- split info table (must include RC_ID, was_split, parent_rc_id) ---
#rc_work["RC_ID"] = rc_work["RC_ID"].astype(str)
split_info = rc_split[["RC_ID", "was_split", "Parent_ID"]]

# 1) attach split info
rc_work = rc_work.merge(split_info, on="RC_ID", how="left")

# # 2) create group_id (parent if split, else own RC_ID)
is_split = rc_work["was_split"].astype(str).str.lower().isin(["true", "1", "yes"])
rc_work["group_id"] = rc_work["RC_ID"]
rc_work.loc[is_split, "group_id"] = rc_work.loc[is_split, "Parent_ID"]

# # 3) aggregate metrics to group level
# group_df = (
#     rc_work.groupby("group_id", dropna=False)
#            .agg(
#                LDC_retailN=("tr_retailN", "sum"),
#                RegionName=("RegionName", "first"),
#                CountyName=("CountyName", "first"),
#                LAD24NM=("LAD24NM", "first"),
#                gsscode=("gsscode", "first"),
#               centre_type=("centre_type", "first")
#            )
#            .reset_index()
#            .rename(columns={"group_id": "RC_ID"})
# )


In [189]:
split_info[split_info['RC_ID']==8]

,RC_ID,was_split,Parent_ID
4922,8,False,NaN


In [186]:
rc_work[rc_work['group_id'] == 8]

,RC_ID,Total_POI_count,H3_cells,Retail,Dining_and_Drinking,Leisure_Travel,Community_Health,category_names,centre_type,LocalAuthorityNM_1,...,LocalAuthorityNM_3,geometry,tr_retailN,LAD24NM,CountyName,RegionName,gsscode,was_split,Parent_ID,group_id
2378,119,678,108,360.0,237.0,65.0,16.0,['Tourist Information and Service' 'Gastropub'...,centre,Islington,...,0,"MULTIPOLYGON (((531523.8433 183367.75465, 5314...",393.0,Islington,Greater london,London,E34004707,True,8.0,8
2379,200,502,106,241.0,196.0,55.0,10.0,"['Electronics Store' ""Doctor's Office"" 'Touris...",centre,Islington,...,0,"MULTIPOLYGON (((530980.53899 183158.83048, 531...",308.0,Islington,Greater london,London,E34004707,True,8.0,8
2380,353,358,79,142.0,165.0,34.0,17.0,"['Latin American Restaurant' ""Doctor's Office""...",centre,Islington,...,0,"MULTIPOLYGON (((531213.21867 185081.15414, 531...",220.0,Islington,Greater london,London,E34004707,True,8.0,8
2381,294,399,77,211.0,163.0,15.0,10.0,['Latin American Restaurant' 'Gastropub' 'Souv...,centre,Islington,...,0,"MULTIPOLYGON (((530484.57927 185894.78405, 530...",299.0,Islington,Greater london,London,E34004707,True,8.0,8
2382,589,248,69,138.0,84.0,14.0,12.0,['Electronics Store' 'Gastropub' 'Souvlaki Sho...,centre,Islington,...,0,"MULTIPOLYGON (((529176.31329 186186.57342, 529...",171.0,Islington,Greater london,London,E34004707,True,8.0,8
2383,1147,137,48,45.0,70.0,15.0,7.0,['Electronics Store' 'Sporting Goods Retail' '...,centre,Islington,...,0,"MULTIPOLYGON (((530797.78893 185321.11428, 530...",73.0,Islington,Greater london,London,E34004707,True,8.0,8
2384,1056,149,38,92.0,40.0,9.0,8.0,"['Electronics Store' ""Doctor's Office"" 'Latin ...",centre,Islington,...,0,"MULTIPOLYGON (((531874.71102 183740.36173, 531...",108.0,Islington,Greater london,London,E34004707,True,8.0,8
2385,1867,78,22,32.0,34.0,8.0,4.0,"[""Doctor's Office"" 'Betting Shop' 'Convenience...",centre,Islington,...,0,"MULTIPOLYGON (((529938.01546 186473.36052, 529...",55.0,Islington,Greater london,London,E34004707,True,8.0,8
2386,1504,101,19,56.0,32.0,8.0,5.0,['Sushi Restaurant' 'Physical Therapy Clinic' ...,centre,Islington,...,0,"MULTIPOLYGON (((531571.51885 184258.77123, 531...",57.0,Islington,Greater london,London,E34004707,True,8.0,8
2387,3050,41,18,11.0,15.0,12.0,3.0,"[""Doctor's Office"" 'Physical Therapy Clinic' '...",centre,Islington,...,0,"MULTIPOLYGON (((531710.37384 182851.75401, 531...",15.0,Islington,Greater london,London,E34004707,True,8.0,8


In [139]:
group_df.sort_values("RC_ID", ascending=True).head(11)

,RC_ID,LDC_retailN,RegionName,CountyName,LAD24NM,gsscode,centre_type
0,1,15690.0,London,Greater london,Westminster,E34004707,centre
1,2,3653.0,Scotland,City of edinburgh,City of Edinburgh,None,centre
2,3,2735.0,South East,East sussex,Brighton and Hove,E34004748,centre
3,4,2464.0,North West,Greater manchester,Manchester,E34005054,centre
4,5,3009.0,Scotland,City of glasgow,Glasgow City,None,centre
5,6,1708.0,Yorkshire and The Humber,West yorkshire,Leeds,E34004684,centre
6,7,1816.0,North West,Merseyside,Liverpool,E34004801,centre
7,8,2700.0,London,Greater london,Islington,E34004707,centre
8,9,2380.0,London,Greater london,Westminster,E34004707,centre
9,10,2491.0,East Midlands,Nottinghamshire,Nottingham,E34004946,centre


In [140]:
group_df['RegionName'].value_counts()

RegionName
London                      1334
South East                  1212
North West                  1195
East of England              856
Yorkshire and The Humber     839
West Midlands                802
Scotland                     784
South West                   739
East Midlands                611
Wales                        458
North East                   399
NI                           237
Name: count, dtype: int64

In [146]:
group_df1 = apply_hierarchical_classification(group_df)
group_df1['Classification'].value_counts()

Classification
Small Local Centre    7033
Small Retail Park     1079
Town Centre            414
Local Centre           385
District Centre        261
Market Town            146
Large Retail Park       83
Major Town Centre       53
Regional Centre         12
Name: count, dtype: int64

In [147]:
group_df1

,RC_ID,LDC_retailN,RegionName,CountyName,LAD24NM,gsscode,centre_type,RGN_rank,CTY_rank,LAD_rank,BUA_rank,tr_retailN,Classification
0,10,2491.0,East Midlands,Nottinghamshire,Nottingham,E34004946,centre,1.0,1.0,1.0,1.0,2491.0,Regional Centre
1,18,1479.0,East Midlands,Leicestershire,Leicester,E34004647,centre,2.0,1.0,1.0,1.0,1479.0,Major Town Centre
2,39,923.0,East Midlands,Northamptonshire,West Northamptonshire,E34004611,centre,3.0,1.0,1.0,1.0,923.0,Major Town Centre
3,93,586.0,East Midlands,Lincolnshire,Lincoln,E34005030,centre,5.0,1.0,1.0,1.0,586.0,Major Town Centre
4,114,629.0,East Midlands,Leicestershire,Charnwood,E34000597,centre,4.0,2.0,1.0,1.0,629.0,Town Centre
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9461,9590,4.0,Yorkshire and The Humber,South yorkshire,Sheffield,E34004969,retail_park,742.0,206.0,101.0,119.0,4.0,Small Retail Park
9462,9597,3.0,Yorkshire and The Humber,Lincolnshire,North Lincolnshire,E34004192,retail_park,753.0,104.0,14.0,8.0,3.0,Small Retail Park
9463,9600,3.0,Yorkshire and The Humber,South yorkshire,Doncaster,E34004696,retail_park,754.0,211.0,45.0,30.0,3.0,Small Retail Park
9464,9603,8.0,Yorkshire and The Humber,West yorkshire,Bradford,E34004684,retail_park,631.0,286.0,66.0,249.0,8.0,Small Retail Park


In [148]:
group_df2 = apply_manual_edits(group_df1)
group_df2['Classification'].value_counts()

Classification
Small Local Centre    7033
Small Retail Park     1079
Town Centre            409
Local Centre           384
District Centre        261
Market Town            147
Large Retail Park       83
Major Town Centre       58
Regional Centre         12
Name: count, dtype: int64

In [149]:
group_df3 = apply_shopping_outlet_overrides(
    group_df2,
    rc_uk_path="geo/postworkshopcentres_withsplit.gpkg",
    rcid_lookup_csv="output/rcid_lookup_old_to_new.csv",
)
group_df3 = apply_out_of_town_sizing(group_df3)
group_df3["Classification"].value_counts()


Classification
Small Local Centre       7014
Small Retail Park        1077
Town Centre               400
Local Centre              379
District Centre           251
Market Town               147
Large Retail Park          83
Major Town Centre          57
Large Shopping Centre      42
Regional Centre            12
Small Shopping Centre       6
Name: count, dtype: int64

In [158]:
#5) classify at group level
group_df = apply_hierarchical_classification(group_df)
group_df = apply_manual_edits(group_df)
group_df = apply_shopping_outlet_overrides(group_df)

# 6) join group classification back to parts
rc_work = rc_work.merge(
    group_df[["RC_ID", "Classification"]],
    left_on="group_id", right_on="RC_ID",
    how="left", suffixes=("", "_group")
)

rc_work["Classification_join"] = rc_work["Classification_group"]

# 7) cleanup helper columns (optional)
rc_work = rc_work.drop(columns=["Classification_group", "group_id", "was_split", "parent_rc_id"], errors="ignore")


In [159]:
rc_work["Classification_join"].value_counts()

Classification_join
Small Local Centre              7014
Small Retail Park               1077
Town Centre                      438
Local Centre                     379
District Centre                  260
Market Town                      147
Major Town Centre                 86
Regional Centre                   84
Large Retail Park                 83
Out of Town Shopping Centres      68
Name: count, dtype: int64

In [ ]:
# wrap up as a function
def classify_using_split_groups1(
    df: pd.DataFrame,
    split_df: pd.DataFrame,
    parent_col: str = "Parent_ID",   # in split_df
    split_col: str = "was_split",       # in split_df
    id_col: str = "RC_ID",
) -> pd.DataFrame:
    df = df.copy()

    # attach split info
    df = df.merge(split_df[[id_col, split_col, parent_col]], on=id_col, how="left")

    # group id: parent if split, else own id
    df["group_id"] = df[id_col]
    df.loc[df[split_col] == True, "group_id"] = df.loc[df[split_col] == True, parent_col]

    # aggregate metrics + keep admin fields for ranking
    group_df = (
        df.groupby("group_id", dropna=False)
          .agg(
              LDC_retailN=("tr_retailN", "sum"),
              #tr_VOA_retailN=("tr_VOA_retailN", "sum") if "tr_VOA_retailN" in df.columns else ("RC_ID", "size"),
              RegionName=("RegionName", "first"),
              CountyName=("CountyName", "first"),
              LAD24NM=("LAD24NM", "first"),
              gsscode=("gsscode", "first"),
              centre_type=("centre_type", "first")
          )
          .reset_index()
          .rename(columns={"group_id": "RC_ID"})
    )

    # classify at group level
    group_df = apply_hierarchical_classification(group_df)
    group_df = apply_manual_edits(group_df)
    group_df = apply_shopping_outlet_overrides(group_df)
    group_df = apply_out_of_town_sizing(group_df)

    # bring group classification back to original rows
    df = df.merge(
        group_df[["RC_ID", "Classification"]],
        left_on="group_id", right_on="RC_ID",
        how="left", suffixes=("", "_group")
    )
    df["Classification_group"] = df['Classification']

    # cleanup
    df = df.drop(columns=["Classification_group", "group_id", split_col, parent_col], errors="ignore")

    return df


In [204]:
def classify_using_group_sums(
    df: pd.DataFrame,
    split_df: pd.DataFrame,
    parent_col: str = "Parent_ID",
    split_col: str = "was_split",
    id_col: str = "RC_ID",
) -> pd.DataFrame:
    df = df.copy()

    # attach split info
    df = df.merge(split_df[[id_col, split_col, parent_col]], on=id_col, how="left")

    # create group_id (string to avoid dtype issues)
    is_split = df[split_col].astype(str).str.lower().isin(["true", "1", "yes"])
    df["group_id"] = df[id_col].astype(str)
    df.loc[is_split, "group_id"] = "P_" + df.loc[is_split, parent_col].astype(str)

    # group sum for all rows in the group
    df["group_LDC_retailN"] = df.groupby("group_id")["tr_retailN"].transform("sum")

    # for unsplit rows, just keep their own tr_retailN explicitly
    df.loc[~is_split, "group_LDC_retailN"] = df.loc[~is_split, "tr_retailN"]

    # use grouped totals for classification
    df["LDC_retailN"] = df["group_LDC_retailN"]

    # classify at row level using grouped totals
    df = apply_hierarchical_classification(df)
    df = apply_manual_edits(df)
    df = apply_shopping_outlet_overrides(df)
    df = apply_out_of_town_sizing(df)

    return df


In [205]:
df_join = classify_using_group_sums(
    df= rc_geo,
    split_df=rc_split
)
df_join['Classification'].value_counts()

Classification
Small Local Centre       7017
Small Retail Park        1077
District Centre           401
Town Centre               399
Local Centre              381
Market Town               147
Large Retail Park          83
Major Town Centre          58
Large Shopping Centre      42
Regional Centre            13
Small Shopping Centre       6
Name: count, dtype: int64

In [ ]:
# df_join = df_join.rename(
#     columns={
#         "Classification": "before_classification",
#         "Classification_join": "Classification",
#     }
# )
# df_join

In [171]:
df_join

,RC_ID,Total_POI_count,H3_cells,Retail,Dining_and_Drinking,Leisure_Travel,Community_Health,category_names,centre_type,LocalAuthorityNM_1,LocalAuthorityNM_2,LocalAuthorityNM_3,geometry,tr_retailN,LAD24NM,CountyName,RegionName,gsscode,RC_ID_group,Classification
0,6025,16,5,7.0,6.0,0.0,3.0,['Asian Restaurant' 'Electronics Store' 'Café'...,centre,North East Derbyshire,0,0,"MULTIPOLYGON (((440445.73886 367018.09139, 440...",2.0,North East Derbyshire,Derbyshire,East Midlands,E34004845,6025,Small Local Centre
1,1193,131,71,86.0,33.0,4.0,8.0,"['Electronics Store' ""Doctor's Office"" 'Drugst...",centre,North East Derbyshire,0,0,"MULTIPOLYGON (((439116.77659 363407.88751, 439...",131.0,North East Derbyshire,Derbyshire,East Midlands,E34004845,1193,Town Centre
2,6644,14,11,11.0,2.0,1.0,0.0,['Electronics Store' 'Hardware Store' 'Conveni...,centre,Cornwall,0,0,"MULTIPOLYGON (((194309.88097 76200.71289, 1943...",10.0,Cornwall,Cornwall,South West,E34001781,6644,Small Local Centre
3,6026,16,44,11.0,3.0,2.0,0.0,['Gym' 'Gastropub' 'Brewery' 'Funeral Home' 'B...,centre,North East Derbyshire,0,0,"MULTIPOLYGON (((439008.41769 364010.85389, 438...",17.0,North East Derbyshire,Derbyshire,East Midlands,E34004845,6026,Small Local Centre
4,7407,12,4,6.0,3.0,1.0,2.0,"['Hardware Store' ""Doctor's Office"" 'Indian Re...",centre,Chesterfield,0,0,"MULTIPOLYGON (((438294.89934 371476.61952, 438...",8.0,Chesterfield,Derbyshire,East Midlands,E34004985,7407,Small Local Centre
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9619,2844,46,16,32.0,13.0,1.0,0.0,['Convenience Store' 'Discount Store' 'Restaur...,retail_park,"Armagh City, Banbridge and Craigavon",0,0,"MULTIPOLYGON (((111614.72348 512113.40565, 111...",59.0,"Armagh City, Banbridge and Craigavon",Armagh,NI,NI,2844,Large Retail Park
9620,4025,29,22,17.0,8.0,2.0,2.0,['Drugstore' 'Pet Service' 'Convenience Store'...,retail_park,Braintree,0,0,"MULTIPOLYGON (((578163.67621 233984.52019, 578...",14.0,Braintree,Essex,East of England,E34004351,4025,Small Retail Park
9621,2676,50,11,34.0,14.0,2.0,0.0,['Textiles Store' 'Electronics Store' 'Sportin...,retail_park,"Bournemouth, Christchurch and Poole",0,0,"MULTIPOLYGON (((401222.30741 90489.38681, 4011...",46.0,"Bournemouth, Christchurch and Poole",Dorset,South West,E34005031,2676,Small Retail Park
9622,3757,32,22,26.0,6.0,0.0,0.0,['Sporting Goods Retail' 'Convenience Store' '...,retail_park,Glasgow City,0,0,"MULTIPOLYGON (((255356.56308 666668.68178, 255...",50.0,Glasgow City,City of glasgow,Scotland,NaN,3757,Large Retail Park


In [206]:
# export
#cols = ["RGN_rank", "CTY_rank", "LAD_rank", "BUA_rank", "tr_retailN"]
cols = ["tr_retailN"]

df_join[cols] = df_join[cols].apply(pd.to_numeric, errors="coerce").round().astype("int64")
df_join["geometry"] = df_join["geometry"].apply(
    lambda geom: wkt.loads(geom) if isinstance(geom, str) else geom
)
out_gdf = gpd.GeoDataFrame(df_join, geometry="geometry", crs="EPSG:27700")
out_path = "output/rc2025_classified_joined_BUA11.gpkg"

#out_gdf.to_file(out_path, driver="ESRI Shapefile")
out_gdf.to_file(out_path, driver="GPKG", layer="rc2025_classified_joined_BUA11")


In [ ]:
cols = ["RGN_rank", "CTY_rank", "LAD_rank", "BUA_rank", "tr_retailN"]
df_join[cols] = df_join[cols].apply(pd.to_numeric, errors="coerce").round().astype("int64")
df_join["geometry"] = df_join["geometry"].apply(
    lambda geom: wkt.loads(geom) if isinstance(geom, str) else geom
)
out_gdf = gpd.GeoDataFrame(df_join, geometry="geometry", crs="EPSG:27700")
out_path = "output/rc2025_classified_joined.shp"

out_gdf.to_file(out_path, driver="ESRI Shapefile")
out_gdf.to_file("output/rc2025_classified_joined.gpkg", driver="GPKG")

# redo the manual edits

In [ ]:
def apply_manual_edits2(
    df: pd.DataFrame,
    rcid_lookup_csv: str = "output/rcid_lookup_old_to_new.csv",
    manual_hierarchy_csv: str | None = None,
    id_col: str = "RC_ID",          # column in df to update (new ids)
    group_col: str | None = None,   # optional: group_id column if you have it
) -> pd.DataFrame:
    """
    Manual classification edits + optional Manual_Hierarchy upgrades.
    Handles one-to-many old -> new RC_ID mapping by applying to all new RC_IDs.
    """
    df = df.copy()

    # --- build old -> list of new mapping ---
    lookup = pd.read_csv(rcid_lookup_csv)
    old_to_new = {}

    for old_id, new_id in zip(lookup["RC_ID"], lookup["new_rcid"]):
        old_to_new.setdefault(old_id, []).append(new_id)

    def map_old_to_new_list(old_id: str):
        # if no mapping exists, fall back to original
        return old_to_new.get(old_id, [old_id])

    # --- Manual classification edits (old RC_IDs in the R code) ---
    manual_classification_old = {
        "RC_EW_2920": "Market Town",
        "RC_SC_332": "Major Town Centre",
        "RC_EW_876": "Major Town Centre",
        "RC_EW_5564": "Major Town Centre",
        "RC_EW_61": "Major Town Centre",
        "RC_EW_3096": "Major Town Centre",
        "RC_EW_3254": "Major Town Centre",
        "RC_EW_3301": "Major Town Centre",
        "RC_EW_3476": "Major Town Centre",
        "RC_EW_3522": "Regional Centre",  # plus Bristol
        "RC_EW_2751": "Regional Centre",  # Liverpool One
        "RC_SC_280": "Regional Centre",   # Edinburgh
    }

    for old_id, cls in manual_classification_old.items():
        new_ids = map_old_to_new_list(old_id)
        mask = df[id_col].isin(new_ids)
        if group_col:
            mask = mask | df[group_col].isin(new_ids)
        df.loc[mask, "Classification"] = cls

    # --- Manual hierarchy list (upgrade Town Centre -> Major Town Centre) ---
    if manual_hierarchy_csv:
        manual_path = Path(manual_hierarchy_csv)
        if manual_path.exists():
            rc_manual = pd.read_csv(manual_path)
            rc_manual = rc_manual.loc[
                rc_manual["H_Classification"] == "Major Town Centre", "RC_ID"
            ].dropna()

            # expand old IDs to all new IDs
            new_ids = []
            for old_id in rc_manual.astype(str).unique():
                new_ids.extend(map_old_to_new_list(old_id))

            mask = df[id_col].isin(new_ids) & (df["Classification"] == "Town Centre")
            if group_col:
                mask = mask | (df[group_col].isin(new_ids) & (df["Classification"] == "Town Centre"))
            df.loc[mask, "Classification"] = "Major Town Centre"

    return df


In [ ]:
df_join2 = df = apply_manual_edits2(df_join, "output/rcid_lookup_old_to_new.csv", group_col="group_id")
df_join2['Classification_join'].value_counts()

# check for the outlets

In [ ]:

# build outlets DataFrame 
outlets = pd.DataFrame(
    [
        ("Regional Shopping Centre", "Westfield London", 51.50768597440922, -0.22088352838801395),
        ("Regional Shopping Centre", "Metrocentre", 54.957214904105754, -1.6674536751658506),
        ("Regional Shopping Centre", "Trafford Centre", 53.465100417291744, -2.347637851106119),
        ("Regional Shopping Centre", "Westfield Stratford City", 51.54425552636472, -0.004332458383472109),
        ("Regional Shopping Centre", "Bluewater", 51.43912203176112, 0.27160383738290644),
        ("Regional Shopping Centre", "intu Merry Hill", 52.48138718476229, -2.110658395511001),
        ("Regional Shopping Centre", "Lakeside Shopping Centre", 51.489406902239956, 0.2853764470235825),
        ("Regional Shopping Centre", "Meadowhall", 53.41411230589209, -1.4111151687565566),
        ("Regional Shopping Centre", "East Kilbride Shopping Centre", 55.76021412051323, -4.17708950101207),
        ("Regional Shopping Centre", "Silverburn Centre", 55.82112954772368, -4.341253330960348),
        ("Regional Shopping Centre", "intu Braehead", 55.87600450967236, -4.364642953985305),
        ("Regional Shopping Centre", "White Rose Centre", 53.75868081616114, -1.5734907774777223),
        ("Large Outlet Shopping Centre", "Ashford Designer Outlet", 51.137387301260496, 0.8801687441992894),
        ("Large Outlet Shopping Centre", "Astle Park West Bromwich", 52.520103339621166, -1.9960259925402608),
        ("Large Outlet Shopping Centre", "Bicester Outlet Village", 51.89237714202758, -1.1560770883204268),
        ("Large Outlet Shopping Centre", "Bideford Atlantic Village", 51.00755545946937, -4.237327761297502),
        ("Large Outlet Shopping Centre", "Burberry Factory Outlet London", 51.54651001026377, -0.05093197052890261),
        ("Large Outlet Shopping Centre", "Castleford Junction 32 Outlet", 53.712463797168354, -1.3388364244282458),
        ("Large Outlet Shopping Centre", "Chatham Dockside Outlet Centre", 51.40160304040388, 0.5377104489852201),
        ("Large Outlet Shopping Centre", "Cheshire Oaks Designer Outlet", 53.26539272808919, -2.8809029974792315),
        ("Large Outlet Shopping Centre", "Clacton Factory Outlet", 51.81858769266023, 1.168209463308007),
        ("Large Outlet Shopping Centre", "Clarks Village Outlet Somerset", 51.12833201493458, -2.741588552823589),
        ("Large Outlet Shopping Centre", "Dalton Park Outlet", 54.81537131857434, -1.3739226162513756),
        ("Large Outlet Shopping Centre", "Doncaster Lakeside Village", 53.50769203205623, -1.1135984485990036),
        ("Large Outlet Shopping Centre", "Dover De Bradelei Wharf Outlet", 51.12211019226935, 1.3141697547138849),
        ("Large Outlet Shopping Centre", "East Midlands Outlet Junction 28", 53.108083884731556, -1.3126583496100666),
        ("Large Outlet Shopping Centre", "Freeport Braintree", 51.870070700810075, 0.5714801267433955),
        ("Large Outlet Shopping Centre", "Freeport Fleetwood Outlet", 53.91786152727727, -3.0118505404061744),
        ("Large Outlet Shopping Centre", "Freeport Talke Stoke-on-Trent", 53.070597467076176, -2.263078253210015),
        ("Large Outlet Shopping Centre", "Gloucester Quays Outlet", 51.85953205593224, -2.2516156614946916),
        ("Large Outlet Shopping Centre", "Hertfordshire Galleria Outlet Shopping", 51.76141229905345, -0.24021860677316503),
        ("Large Outlet Shopping Centre", "Hornsea Freeport Outlet", 53.898485374967684, -0.1724229262407349),
        ("Large Outlet Shopping Centre", "Kilver Court Designer Village", 51.19063584650441, -2.5353243560244905),
        ("Large Outlet Shopping Centre", "London Designer Outlet Wembley", 51.55646778421662, -0.28320440479433673),
        ("Large Outlet Shopping Centre", "Lowry Outlet Salford Quays", 53.47092253822526, -2.292514874626797),
        ("Large Outlet Shopping Centre", "Newcastle Royal Quays Outlet", 54.99532901611218, -1.4642812945442476),
        ("Large Outlet Shopping Centre", "Portsmouth Gunwharf Quays", 50.79546395254288, -1.103824062300293),
        ("Large Outlet Shopping Centre", "Springfields Spalding Outlet", 52.799438891276665, -0.126329033767297),
        ("Large Outlet Shopping Centre", "Swindon Designer Outlet", 51.56272470333755, -1.7971899666540831),
        ("Large Outlet Shopping Centre", "The Mill Outlet Batley", 53.7129199988173, -1.6251337290018772),
        ("Large Outlet Shopping Centre", "York Designer Outlet", 53.92233415376806, -1.0768705880781113),
        ("Large Outlet Shopping Centre", "Banbridge Village Outlet", 54.33172185692118, -6.276147653115943),
        ("Large Outlet Shopping Centre", "Cribbs Causeway", 51.524552488774034, -2.5955220725284307),
        ("Large Outlet Shopping Centre", "Bridgend Designer Outlets Wales", 51.532410963682594, -3.576839020076668),
        ("Large Outlet Shopping Centre", "Gretna Gateway Outlet Village", 54.99688074646182, -3.056523264134829),
        ("Large Outlet Shopping Centre", "Livingston Designer Outlet", 55.88310933797727, -3.520137944032474),
        ("Large Outlet Shopping Centre", "Sterling Mills Outlet", 56.14927287877639, -3.740053433726151),
    ],
    columns=["SubType", "Name", "Lat", "Lon"],
)
outlets["Type"] = "Out of Town Shopping Centres"

# Make GeoDataFrame
outlets_gdf = gpd.GeoDataFrame(
    outlets,
    geometry=gpd.points_from_xy(outlets["Lon"], outlets["Lat"]),
    crs="EPSG:4326",
)

# Export
outlets_gdf.to_file("output/outlets.shp", driver="ESRI Shapefile")
